![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 30 -- Lab 2: Brain Scans with Fine-Tuning

An Olympiad-style image task far from ImageNet: 7,200 Kaggle brain MRI scans in four classes (glioma, meningioma, no tumour, pituitary), scored by accuracy on a separate Testing folder. You build the pipeline with the ImageNet contract, probe the frozen encoder first (linear probing, two epochs), then thaw the encoder with a ladder of learning rates and fine-tune for three more epochs, read whether the second phase added anything, score the Testing folder with a confusion matrix, look at the mistakes, and write a submission file.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `masoudnickparvar/brain-tumor-mri-dataset` (about 160 MB), downloaded by `kagglehub` in the setup cell: `Training/<class>/*.jpg` (1,400 scans per class) and `Testing/<class>/*.jpg` (400 per class). Many scans are stored as grey images of different sizes; `ImageFolder` converts them to three channels. You hold out 20% of Training for validation; Testing is the honest score at the end. Use a GPU (Runtime, Change runtime type, T4): the five epochs take about 10 minutes there.

---
# Setup

In [ ]:
# kagglehub fetches the Kaggle dataset; Colab has everything else.
!pip install -q kagglehub

In [ ]:
import os, random, time
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import confusion_matrix
random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
TRAIN_DIR, TEST_DIR = os.path.join(path, "Training"), os.path.join(path, "Testing")
CLASSES = sorted(os.listdir(TRAIN_DIR))
print(CLASSES)
print("Training:", {c: len(os.listdir(os.path.join(TRAIN_DIR, c))) for c in CLASSES})
print("Testing: ", {c: len(os.listdir(os.path.join(TEST_DIR, c))) for c in CLASSES})

---
# Part 1 -- Look, then build the pipeline

The *How Far From ImageNet* slide: scans are grey, come in many sizes, and show tissue textures ImageNet never had. Look first, then the same pipeline as the flowers.

## Task 1: Look at the scans

Show two scans per class in a 2 x 4 grid: for each class take the first two file names of `sorted(os.listdir(...))`, open them with `Image.open(...)`, and print each image's `.size` and `.mode` as its title (`fontsize=8`), axes off. How many of the eight are grey (`mode` `L`), and do the sizes agree?

In [ ]:
# Your code here


## Task 2: The transforms and the three ImageFolders

Create `normalize` with ImageNet's mean and std, `train_tf` (`RandomResizedCrop(224, scale=(0.7, 1.0))`, `RandomHorizontalFlip()`, `ToTensor()`, `normalize`) and `val_tf` (`Resize(256)`, `CenterCrop(224)`, `ToTensor()`, `normalize`). Build `full_train = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)`, `full_val = datasets.ImageFolder(TRAIN_DIR, transform=val_tf)` and `test_ds = datasets.ImageFolder(TEST_DIR, transform=val_tf)`. Print `full_train.classes` and the three lengths. `ImageFolder` opens every file as RGB, so the grey scans become three identical channels. A scan cropped to 70% of its area still shows the tumour, and a left-right flip keeps the label.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert full_train.classes == CLASSES and len(full_train) == 5600 and len(test_ds) == 1600, 'ImageFolder: four classes, 5,600 training scans, 1,600 test scans'
print('Task 2 passed')

## Task 3: Split and DataLoaders

Shuffle `idx = list(range(len(full_train)))` with `random.shuffle`, take the first 1,120 indices (20%) as `val_idx` and the rest as `train_idx`. Build `train_loader` (`Subset(full_train, train_idx)`, `batch_size=32, shuffle=True, num_workers=2`), `val_loader` (`Subset(full_val, val_idx)`, `batch_size=64, shuffle=False, num_workers=2`) and `test_loader` (`test_ds`, `batch_size=64, shuffle=False, num_workers=2`). Take one training batch and print its shape, mean and std.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(train_idx) == 4480 and len(val_idx) == 1120 and tuple(xb.shape) == (32, 3, 224, 224), 'split 4,480 / 1,120; a batch of (32, 3, 224, 224)'
print('Task 3 passed')

---
# Part 2 -- Phase 1: probe the frozen encoder

The *Recipe That Keeps the Knowledge* slide: linear probing first. The probe's accuracy tells you how far these scans are from ImageNet, and gives the head a sensible start before anything else moves.

## Task 4: Load, freeze, new head

Create `model = resnet18(weights=ResNet18_Weights.DEFAULT)`, freeze every parameter, replace `model.fc` by `nn.Linear(model.fc.in_features, len(CLASSES))`, move it to `device`, and create `criterion = nn.CrossEntropyLoss()` and `optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)`. Print the trainable count, `sum(p.numel() for p in model.parameters() if p.requires_grad)`: 512 x 4 + 4 = 2,052.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_trainable == 2052, 'frozen encoder + Linear(512, 4): 2,052 trainable parameters'
print('Task 4 passed')

## Task 5: The evaluate function and the loss before training

Write `evaluate(model, loader)` as in Day 29's Lab 2: `eval()` mode, `torch.no_grad()`, the summed loss and the number of correct predictions over the loader, returning the mean loss and the accuracy. Run it on `val_loader` before training and print both numbers: four classes and a random head give a loss near ln 4 = 1.39 and an accuracy near 0.25.

Syntax hint (the shape of the call, not the answer):

```python
def evaluate(model, loader):
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            ...
    return mean_loss, accuracy
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert 0.05 < acc_before < 0.5, 'evaluate before training: a random head scores near chance, 0.25'
print('Task 5 passed')

## Task 6: The probe: 2 epochs, the loop written out

Train the head for 2 epochs with the full loop: `model.train()`, the batches moved to `device`, `loss = criterion(model(xb), yb)`, the four moves, a running total for the mean training loss; after the batches `evaluate(model, val_loader)`, append `(train_loss, val_loss, val_acc)` to `history`, print the numbers with the seconds, and save `torch.save(model.state_dict(), "best.pt")` whenever `val_acc` beats `best_acc`. Then store `probe_acc = best_acc`: this is the probe's answer about how far scans are from ImageNet.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(history) == 2 and probe_acc > 0.5 and os.path.exists('best.pt'), 'two probe epochs: a validation accuracy above 0.5 and a best.pt checkpoint'
print('Task 6 passed')

---
# Part 3 -- Phase 2: thaw the encoder with a ladder of learning rates

The *Discriminative Learning Rates* and *How Far From ImageNet* slides: scans are far from ImageNet, so more than the last stage should move; the deeper the stage, the smaller its step. `conv1` and `bn1`, the general first layer, stay frozen.

## Task 7: Thaw with the ladder

Set `param.requires_grad = True` for every parameter of `model.layer1`, `layer2`, `layer3` and `layer4` (leave `conv1` and `bn1` frozen). Build `groups`: start with `lr = 1e-3` and loop over `[model.fc, model.layer4, model.layer3, model.layer2, model.layer1]`, appending `{"params": part.parameters(), "lr": lr}` and dividing `lr` by 2.6 each time. Create a new `optimizer = torch.optim.Adam(groups)`. Print the trainable count and the five learning rates rounded with `f"{g['lr']:.1e}"`.

Syntax hint (the shape of the call, not the answer):

```python
groups, lr = [], 1e-3
for part in [...]:
    groups.append({"params": part.parameters(), "lr": lr})
    lr /= 2.6
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_trainable == 11_169_028 and len(optimizer.param_groups) == 5 and abs(optimizer.param_groups[1]['lr'] - 1e-3 / 2.6) < 1e-9, 'phase 2: 11,169,028 trainable (everything but conv1 and bn1), five groups, layer4 at 1e-3 / 2.6'
print('Task 7 passed')

## Task 8: Fine-tune: 3 more epochs

Run the same written-out loop for 3 more epochs, appending to the same `history` and keeping the same `best_acc` and `best.pt` logic, so the checkpoint is the best epoch of either phase. Print the numbers with the seconds; the time per epoch grows now that the gradient flows through the encoder. Store `finetune_acc = best_acc` and print it next to `probe_acc`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(history) == 5 and finetune_acc >= probe_acc and finetune_acc > 0.85, 'five epochs in all; the best accuracy after fine-tuning should be above 0.85 and not below the probe'
print('Task 8 passed')

## Task 9: Read the curves

Plot two panels from `history` (epochs 1 to 5): training and validation loss on the left, validation accuracy on the right, each with a dashed vertical line at 2.5 marking where the encoder was thawed, titles, labels and a legend. Did phase 2 add points, and is the validation loss still falling at epoch 5?

In [ ]:
# Your code here


---
# Part 4 -- The honest score, the mistakes, the submission

The Testing folder was never seen: load the best checkpoint, score it, look, write the file.

## Task 10: Testing accuracy and the confusion matrix

Load `best.pt` back with `model.load_state_dict(torch.load("best.pt", map_location=device))`, collect `y_pred` and `y_true` over `test_loader` under `torch.no_grad()`, print the test accuracy, then `cm = confusion_matrix(y_true, y_pred)` drawn with `imshow`, the class names on both axes and the count in every cell, and the per-class accuracy `cm.diagonal() / cm.sum(axis=1)`. Which two classes are confused most?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert cm.sum() == 1600 and test_acc > 0.8, 'the confusion matrix counts all 1,600 test scans and the test accuracy is above 0.8'
print('Task 10 passed')

## Task 11: Look at the mistakes

Show the first 8 test scans where `y_pred != y_true` with the true and predicted class as titles. `test_ds[i]` gives the normalised tensor; undo the normalisation for display as in Day 29 (`t * std + mean`, `permute(1, 2, 0)`, `clamp(0, 1)`). Do the mistakes look like the pair the confusion matrix pointed at?

In [ ]:
# Your code here


## Task 12: Write the submission

Build a DataFrame with `image`, the file name of each test scan in the loader's order (`os.path.basename(p) for p, _ in test_ds.samples`), and `label`, the predicted class name. Save it as `submission.csv` without the index, read it back and print its shape and first five rows.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert list(check.columns) == ['image', 'label'] and len(check) == 1600 and set(check['label']) <= set(CLASSES), 'submission.csv: 1,600 rows, columns image and label, labels among the four class names'
print('Task 12 passed')

---
## Task 13: Reflect

Task 6 printed the probe's accuracy and Task 8 the fine-tuned one: how many points did thawing the encoder add, and how does that compare with the flowers in Lab 1, where the probe was already strong? Explain the difference with the *How Far From ImageNet* slide. Task 7 gave `layer1` a learning rate 46 times smaller than the head's: what would go wrong at 1e-3 for every stage, in the words of the *Math* slide? Your validation split came from the Training folder, which contains augmented copies of some scans: why can the validation accuracy be higher than the Testing accuracy, and which of the two would you report? If Friday's budget allowed only two epochs in total, which phase would you keep?

*Your answers here*

---
## Conclusion

- **The pipeline on a far domain**: grey scans of many sizes through `ImageFolder` with the ImageNet contract and label-preserving augmentation, one split for validation, the Testing folder kept honest.
- **Probe first**: the frozen encoder with a `Linear(512, 4)` head, two epochs, the accuracy read as a measure of how far scans are from ImageNet.
- **Then fine-tune with a ladder**: `layer1` to `layer4` thawed, five parameter groups with learning rates falling by 2.6 per stage, three epochs, the checkpoint shared across both phases.
- **Score, look, submit**: the test accuracy, the confusion matrix and per-class accuracy, the misclassified scans, and a submission with one row per scan.

---

Made By **Sattam Altwaim**